In [1]:
import matplotlib.pyplot as plt
import matplotlib
import sys
import numpy as np
import subprocess
import re

In [2]:

class Polygon:
    def __init__(self, points):
        self.points = points
        if len(points)>0:
            self.points.append(points[0])
            self.xs = [point[0] for point in self.points]
            self.ys = [point[1] for point in self.points]
        else:
            self.xs = []
            self.ys = []

class Frame:
    def __init__(self, frame_points):
        self.frame = frame_points
        self.frame.append(frame_points[0])
        self.xs = [point[0] for point in self.frame]
        self.ys = [point[1] for point in self.frame]

class State:
    def __init__(self, frame, used_polys):
        self.frame = frame
        self.used_polys = used_polys

        
def get_cpp_state(indices):
    cpp_program = './state'  # Path to the compiled C++ program
    arguments = [str(index) for index in indices]

    # Run the C++ program with arguments using subprocess
    result = subprocess.run([cpp_program] + arguments, capture_output=True, text=True)
    
    # Print the output from the C++ program
    out = result.stdout
    states = out.splitlines()
    states_list = []

    for state in states:
        
        # Find the frame
        frame_points = []
        frame_txt = re.findall(r'\{([^\{]+)\}', state)[0]
        points = re.findall(r'\(([^\)]+)\)', frame_txt)
        for point in points:
            x, y = point.split(",")
            frame_points.append([float(x), float(y)])

        frame = Frame(frame_points)

        # Find all the used polygons
        used_polygons = re.findall(r'\[([^\]]+)\]', state)
        used_polygons_list = []
        for used_polygon in used_polygons:
            
            points = re.findall(r'\(([^\)]+)\)', used_polygon)
            points_list = []
            for point in points:
                x, y = point.split(",")
                points_list.append([float(x),float(y)])

            used_polygons_list.append(Polygon(points_list))
                
        states_list.append(State(frame, used_polygons_list))
    return states_list

In [3]:

# Define the callback function to be run on click
def on_click(event):
    ax = event.inaxes  # The axis that was clicked
    if ax is not None:
        # Get the subplot number
        
        subplot_number = ax.get_subplotspec().num1
        print("A")
        indices.append(subplot_number)
        plt.clf()
        states_list = get_cpp_state(indices)
        plot_states(states_list)


def plot_states(states_list):
    global fig, axs

    if len(states_list) != 1:
        sqrt = int(np.sqrt(len(states_list)))+1
        fig, axs = plt.subplots(sqrt, sqrt, figsize=(8,8))
        for i in range(sqrt):
            for j in range(sqrt):
                axs[i,j].axis('off')
    else:
        sqrt = 1
        fig, axs = plt.subplots(sqrt, sqrt, figsize=(8,8))
        axs.axis('off')
        
    fig.canvas.mpl_connect('button_press_event', on_click)

    plt.subplots_adjust(wspace=0.1, hspace=0.1)
    num_polys = 14
    cmap = matplotlib.colormaps["rainbow"]
    colors = [cmap(i) for i in np.linspace(0, 1, num_polys)]
    
    for n, state in enumerate(states_list):

        if sqrt == 1:
            ax = axs
        else:
            ax = axs[n%sqrt, n//sqrt]
        ax.set_xlim([-0.5,12.5])
        ax.set_ylim([-0.5,12.5])
        
        for pp, poly in enumerate(state.used_polys):
            ax.fill(poly.xs, poly.ys, color=colors[pp])
    
        xs, ys = state.frame.xs, state.frame.ys
    
        ax.plot(xs, ys, color='k')
        for x, y, xn, yn in zip(xs[:-1], ys[:-1], xs[1:], ys[1:]):
            ax.arrow(x, y, (xn-x)/2, (yn-y)/2, head_width = 0.4, head_length = 0.4, fc='k', ec='k')
            
    plt.draw()  # Redraw the figure with the new subplots


In [6]:
%matplotlib qt

indices = []
states_list = get_cpp_state([])
plot_states(states_list)

A
A
A
A
A
A


Traceback (most recent call last):
  File "/usr/lib/python3.13/site-packages/matplotlib/cbook.py", line 361, in process
    func(*args, **kwargs)
    ~~~~^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_1193787/2105260206.py", line 12, in on_click
    plot_states(states_list)
    ~~~~~~~~~~~^^^^^^^^^^^^^
  File "/tmp/ipykernel_1193787/2105260206.py", line 23, in plot_states
    axs[i,j].axis('off')
    ~~~^^^^^
TypeError: 'Axes' object is not subscriptable


In [ ]:

        

# Function to update the current figure to a 4x4 grid without creating a new figure
def update_to_4x4_grid():
    
    plt.clf()
    # Create new 4x4 grid in place of the old subplots
    fig.subplots_adjust(hspace=0.5, wspace=0.5)  # Adjust spacing between subplots
    axs = fig.subplots(4, 4)  # Create a new 4x4 grid on the existing figure
    
    # Plot something different on each subplot
    for i, ax in enumerate(axs.flat):
        ax.plot([0, 1, 2], [0, 1, i])  # Different plot for each subplot
        ax.set_title(f'Plot {i + 1}')
        
    plt.draw()  # Redraw the figure with the new subplots

# Function to create the initial 3x3 grid of subplots
def create_initial_3x3_grid():
    global fig, axs
    fig, axs = plt.subplots(3, 3)
    
    # Plot something on each subplot
    for i, ax in enumerate(axs.flat):
        ax.plot([0, 1, 2], [0, 1, i])  # Different plot for each subplot
        ax.set_title(f'Plot {i + 1}')
    
    # Connect the 'button_press_event' to the on_click function
    fig.canvas.mpl_connect('button_press_event', on_click)
    
    # Display the plot
    plt.tight_layout()
    plt.show()

# Start with a 3x3 grid
create_initial_3x3_grid()

In [36]:
%matplotlib qt

# Define the callback function to be run on click
def on_click(event):
    ax = event.inaxes  # The axis that was clicked
    if ax is not None:
        # Get the subplot number
        subplot_number = ax.get_subplotspec().num1 + 1
        
        # Print the clicked subplot number
        print(f'You clicked on Plot {subplot_number}')
        
        # If the user clicked on Plot 6 (which is index 5, because indices start from 0)
        if subplot_number == 6:
            print("You clicked on Plot 6. Changing to a 4x4 subplot grid...")
            update_to_4x4_grid()

# Function to update the current figure to a 4x4 grid without creating a new figure
def update_to_4x4_grid():
    
    plt.clf()
    # Create new 4x4 grid in place of the old subplots
    fig.subplots_adjust(hspace=0.5, wspace=0.5)  # Adjust spacing between subplots
    axs = fig.subplots(4, 4)  # Create a new 4x4 grid on the existing figure
    
    # Plot something different on each subplot
    for i, ax in enumerate(axs.flat):
        ax.plot([0, 1, 2], [0, 1, i])  # Different plot for each subplot
        ax.set_title(f'Plot {i + 1}')
        
    plt.draw()  # Redraw the figure with the new subplots

# Function to create the initial 3x3 grid of subplots
def create_initial_3x3_grid():
    global fig, axs
    fig, axs = plt.subplots(3, 3)
    
    # Plot something on each subplot
    for i, ax in enumerate(axs.flat):
        ax.plot([0, 1, 2], [0, 1, i])  # Different plot for each subplot
        ax.set_title(f'Plot {i + 1}')
    
    # Connect the 'button_press_event' to the on_click function
    fig.canvas.mpl_connect('button_press_event', on_click)
    
    # Display the plot
    plt.tight_layout()
    plt.show()

# Start with a 3x3 grid
create_initial_3x3_grid()

You clicked on Plot 6
You clicked on Plot 6. Changing to a 4x4 subplot grid...
